# Location Dimension Loader

This notebook maintains the `warehouse.dim_location` dimension table.

**Purpose**: Track job posting locations with geographic hierarchy

**Key Features**:
* Stable surrogate keys for locations
* Geographic hierarchy (country, state, city)
* Normalized location data
* Remote work type handling

**Source**: `silver.silver_jobs_current`

**Target**: `workspace.warehouse.dim_location`

In [0]:
%sql
-- Extract distinct locations from silver layer
CREATE OR REPLACE TEMP VIEW location_extract AS
SELECT DISTINCT
  location_norm,
  remote_type,
  -- Parse location components (simplified - in production use geocoding service)
  CASE 
    WHEN location_norm LIKE '%,%,%' THEN SPLIT(location_norm, ',')[2]
    WHEN location_norm LIKE '%,%' THEN SPLIT(location_norm, ',')[1]
    ELSE 'Unknown'
  END as country,
  CASE 
    WHEN location_norm LIKE '%,%,%' THEN SPLIT(location_norm, ',')[1]
    WHEN location_norm LIKE '%,%' THEN SPLIT(location_norm, ',')[0]
    ELSE NULL
  END as state_province,
  CASE 
    WHEN location_norm LIKE '%,%,%' THEN SPLIT(location_norm, ',')[0]
    ELSE location_norm
  END as city,
  CASE 
    WHEN remote_type = 'REMOTE' THEN TRUE
    ELSE FALSE
  END as is_remote,
  COUNT(*) OVER (PARTITION BY location_norm) as job_count
FROM workspace.silver.silver_jobs_current
WHERE location_norm IS NOT NULL;

In [0]:
%sql
-- Generate or maintain stable surrogate keys
CREATE OR REPLACE TEMP VIEW location_with_keys AS
SELECT
  COALESCE(d.location_sk, ROW_NUMBER() OVER (ORDER BY l.location_norm) + COALESCE(max_sk, 0)) as location_sk,
  l.location_norm as location_name,
  TRIM(l.city) as city,
  TRIM(l.state_province) as state_province,
  TRIM(l.country) as country,
  l.remote_type,
  l.is_remote,
  TRUE as is_active,
  CURRENT_TIMESTAMP() as updated_at
FROM location_extract l
LEFT JOIN workspace.warehouse.dim_location d
  ON l.location_norm = d.location_name
CROSS JOIN (
  SELECT COALESCE(MAX(location_sk), 0) as max_sk 
  FROM workspace.warehouse.dim_location
) max_keys;

In [0]:
%sql
-- Merge into target dimension (SCD Type 1)
MERGE INTO workspace.warehouse.dim_location target
USING location_with_keys source
ON target.location_name = source.location_name
WHEN MATCHED THEN UPDATE SET
  target.city = source.city,
  target.state_province = source.state_province,
  target.country = source.country,
  target.remote_type = source.remote_type,
  target.is_remote = source.is_remote,
  target.is_active = source.is_active,
  target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
  location_sk,
  location_name,
  city,
  state_province,
  country,
  remote_type,
  is_remote,
  is_active,
  updated_at
) VALUES (
  source.location_sk,
  source.location_name,
  source.city,
  source.state_province,
  source.country,
  source.remote_type,
  source.is_remote,
  source.is_active,
  source.updated_at
);

In [0]:
%sql
-- Validate location dimension
SELECT 
  COUNT(*) as total_locations,
  COUNT(DISTINCT country) as countries,
  COUNT(DISTINCT state_province) as states,
  COUNT(DISTINCT city) as cities,
  SUM(CASE WHEN is_remote THEN 1 ELSE 0 END) as remote_locations
FROM workspace.warehouse.dim_location;

-- Sample locations
SELECT 
  location_sk,
  location_name,
  city,
  state_province,
  country,
  remote_type,
  is_active
FROM workspace.warehouse.dim_location
ORDER BY country, state_province, city
LIMIT 20;